# Search by cosine similarity in NumPy

A search engine compares a query with every document and returns the closest ones.
Here each document is a vector of word counts, so a long document has large entries even
when it is about the same thing as a short one. Cosine similarity ignores length, and you
compute it for every query and document pair at once, then pick the best matches.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# Counts of four words in each text: "matrix", "vector", "recipe", "oven".
vocab = ["matrix", "vector", "recipe", "oven"]
docs = np.array([
    [4, 3, 0, 0],    # 0  a short note on linear algebra
    [40, 30, 0, 1],  # 1  a long chapter on the same topic
    [0, 1, 5, 4],    # 2  a baking post
    [0, 0, 0, 0],    # 3  an empty page
    [1, 0, 6, 6],    # 4  another baking post
    [3, 5, 1, 0],    # 5  vectors, with a passing recipe joke
], dtype=float)
queries = np.array([
    [1, 1, 0, 0],    # someone looking for linear algebra
    [0, 0, 1, 1],    # someone looking for baking
], dtype=float)
print("documents", docs.shape, " queries", queries.shape)

## Part 1 (coding task): Every cosine at once

Write `cosine_matrix(Q, D)`. `Q` holds $m$ query vectors as rows and `D` holds $n$
document vectors as rows, all of length $d$. Return the $m \times n$ array whose entry
`[i, j]` is the cosine similarity of `Q[i]` and `D[j]`:

$$
\frac{\mathbf{q}_i \cdot \mathbf{d}_j}{\lVert \mathbf{q}_i \rVert\,\lVert \mathbf{d}_j \rVert} .
$$

An all-zero row has no direction. Give it similarity 0 with everything rather than
dividing by zero (which would give `nan`). Use whole-array operations rather than a loop
over pairs: divide each row by its length, then one matrix product gives every dot product.

In [ ]:
def cosine_matrix(Q, D):
    # TODO: scale every row of Q and of D to length 1 (leave zero rows at zero),
    #       then compute all the dot products with one matrix product.
    raise NotImplementedError

In [ ]:
q = np.array([[1, 2, 2], [0, 4, 3]], dtype=float)
d = np.array([[2, 4, 4], [0, 3, 4], [4, 0, -3], [0, 0, 0], [-2, 1, 0], [10, 10, 0]], dtype=float)
S = cosine_matrix(q, d)
assert np.shape(S) == (2, 6), f"expected shape (2, 6), one row per query and one column per document, got {np.shape(S)}"
assert not np.isnan(S).any(), "the all-zero document gives nan: give zero rows similarity 0"
assert np.allclose(cosine_matrix(np.zeros((2, 3)), d), np.zeros((2, 6))), "an all-zero query row has no direction either: give it similarity 0 with every document"
expected = np.array([[1, 14 / 15, -2 / 15, 0, 0, 1 / np.sqrt(2)],
                     [14 / 15, 24 / 25, -9 / 25, 0, 4 / (5 * np.sqrt(5)), 2 * np.sqrt(2) / 5]])
assert not np.allclose(S, q @ d.T), "these are plain dot products: divide by both lengths"
scaled = d * np.array([[1], [3], [0.5], [2], [7], [0.1]])   # stretch each document by its own factor
assert np.allclose(cosine_matrix(q, scaled), cosine_matrix(q, d)), "stretching a document changed its cosine: divide each row by its own length (axis=1), not each column"
assert np.allclose(S, expected), f"some cosines are wrong:\n{np.round(S, 4)}\nexpected\n{np.round(expected, 4)}"
print(np.round(cosine_matrix(queries, docs), 3))
print("Part 1 passed")

## Part 2 (coding task): The best matches

Write `top_k(Q, D, k)`. It returns an integer array of shape $(m, k)$ whose row `i` holds
the indices of the `k` documents with the highest cosine similarity to `Q[i]`, the most
similar first. Use your `cosine_matrix`. `np.argsort` sorts from smallest to largest.

In [ ]:
def top_k(Q, D, k):
    # TODO: rank the documents for every query at once and keep the best k.
    raise NotImplementedError

In [ ]:
q = np.array([[1, 2, 2], [0, 4, 3]], dtype=float)
d = np.array([[2, 4, 4], [0, 3, 4], [4, 0, -3], [0, 0, 0], [-2, 1, 0], [10, 10, 0]], dtype=float)
T = np.asarray(top_k(q, d, 3))
assert T.shape == (2, 3), f"expected shape (2, 3), one row per query, got {T.shape}"
assert np.issubdtype(T.dtype, np.integer), "return document indices, not similarities"
assert T[0, 0] != 2, "these are the least similar documents: argsort sorts in increasing order"
assert not np.array_equal(T[0], [5, 1, 0]), "the best matches are in reverse order: put the most similar first"
assert not np.array_equal(T[0], [5, 0, 1]), "document 5 is first because it is long: rank by cosine, not by dot product"
assert np.array_equal(T, [[0, 1, 5], [1, 0, 5]]), f"got {T.tolist()}, expected [[0, 1, 5], [1, 0, 5]]"
assert np.array_equal(np.asarray(top_k(q, d, 1)), [[0], [1]]), "with k = 1 return the single best document of each query"
print(top_k(queries, docs, 2))
print("Part 2 passed")

## Solution

### Part 1: Every cosine at once

In [ ]:
def unit_rows(X):
    lengths = np.sqrt((X ** 2).sum(axis=1, keepdims=True))   # one length per row, shape (rows, 1)
    return np.divide(X, lengths, out=np.zeros_like(X), where=lengths > 0)

def cosine_matrix(Q, D):
    Q = np.asarray(Q, dtype=float)
    D = np.asarray(D, dtype=float)
    return unit_rows(Q) @ unit_rows(D).T

**Why it works.** 
After every row is divided by its length, the dot product of two rows is
already their cosine, and `U_q @ U_d.T` computes all $m \cdot n$ dot products in one
product (entry `[i, j]` is row `i` of the first times column `j` of the second, which is row
`j` of `U_d`). `keepdims=True` keeps the lengths as a column of shape (rows, 1), so the division
broadcasts across each row. `np.divide` with `where=lengths > 0` leaves zero rows at zero,
so every cosine with the empty page is 0. In the library, the short note and the long chapter
both score about 0.99 against the linear algebra query, though the chapter's counts are ten times
larger.

### Part 2: The best matches

In [ ]:
def top_k(Q, D, k):
    S = cosine_matrix(Q, D)
    order = np.argsort(-S, axis=1, kind="stable")   # largest cosine first, in every row
    return order[:, :k]

**Why it works.** 
Negating the similarities turns `argsort`'s increasing order into decreasing order, and
`axis=1` sorts each query's row on its own. The first `k` columns are the best matches. A
plain dot product would put the long document 5 of the check first for both queries, only
because its counts are large. In the library, the linear algebra query gets the short note and
the long chapter, and the baking query the two baking posts.